> **Series**: *Bayesian Modelling for Industrial Applications*, Part 3. [Part 1](https://sambaiga.github.io/pages/blog/posts/2025/10/2025-10-10-bayesion-foundation.html) introduced Bayesian thinking, and [Part 2](https://sambaiga.github.io/pages/blog/posts/2025/12/2025-12-1-bayesian-regression..html) fitted one Beta regression to the six cells of one chemistry.

An operator of a battery fleet has to decide at every inspection whether to retire a cell or to leave it in service. A cell that has fallen below 80% of its nominal capacity is at the end of its useful life. A cell that stays in service beyond that point risks a failure, and a cell that is retired early wastes the capacity that it still had. The operator does not see the state of health directly, only diagnostics measured in the cycle, and a manufacturer that introduces a new chemistry has tested only a few cells of it. Part 2 gave the six cells of one chemistry the same degradation curve. That was a sensible first model, and its test showed the limit: predicted from the other five, four cells were followed closely and two were missed. Cells differ, and cells of one chemistry resemble each other more than they resemble cells of another, so the question is how far a new cell should be believed to resemble the cells that have been seen.

This post answers it with a hierarchical model, in which each cell draws its parameters from its chemistry and each chemistry from the whole fleet, and it measures the answer in the currency of the operator. The model is built with PyMC[^pymc] and fitted to the CALCE cycling data of Part 2, now with both chemistries. It is judged against its two obvious alternatives, one curve for all cells and one curve per cell, on two cells that the model never saw, and against a control that has no diagnostics at all. The hierarchy separates the two chemistries sharply. As a predictor for a new cell it helps in one setting only, after the cell has shown its first 100 cycles, and the final sections turn each strategy into a retirement rule and compare what the rules cost.

[^pymc]: O. Abril-Pla *et al.*, "PyMC: A modern, and comprehensive probabilistic programming framework in Python," *PeerJ Comput. Sci.*, vol. 9, p. e1516, 2023.

In [1]:
#| output: false
#| code-summary: "Show the setup of the notebook"
import os
import warnings
from importlib.metadata import version
from pathlib import Path
import tempfile

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "2")

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
from sklearn.preprocessing import StandardScaler

from bayes import decision, priors
from bayes.data import calce
from bayes.plot import change, concepts, hierarchy, tables
from bayes.plot.inline import show
from bayes.regression import evaluate
from bayes.regression import hierarchical as hier

FAST = os.environ.get("HIER_FAST") == "1"  # a smoke test with a few draws, to check that the notebook runs
CACHE = Path("cache-fast" if FAST else "cache")
CACHE.mkdir(exist_ok=True)
FEATURES = ["charge_current_auc", "charge_current_mean", "discharge_voltage_auc", "discharge_voltage_crest"]
HELD_OUT = ["CS2_38", "CX2_38"]  # the last cell of each chemistry, kept back for the tests of new cells
SEEN = (0, 100, 300)  # cycles of a new cell that the model has seen
GAP = 50  # embargo, in cycles, between the cycles seen and the cycles scored
MAIN = {"draws": 40 if FAST else 500, "tune": 40 if FAST else 750, "chains": 2 if FAST else 4}
REFIT = {"draws": 40 if FAST else 500, "tune": 40 if FAST else 750, "chains": 2 if FAST else 4}
THRESHOLD = evaluate.END_OF_LIFE  # a cell below this state of health, in percent, is at its end of life
RATIOS = np.round(np.arange(0.02, 0.96, 0.02), 2)  # cost-loss ratios of the decision
PHI_PRIORS = {  # log-normal priors of the precision: the mean and standard deviation of its logarithm
    "Narrow": (float(np.log(100.0)), 0.3),  # centred on 100
    "Default": (float(np.log(100.0)), 1.0),  # centred on 100
    "Wide": (float(np.log(1000.0)), 1.5),  # centred on 1,000
}
print("pymc", version("pymc"), "| arviz", version("arviz"), "| fast mode:", FAST)

pymc 5.25.1 | arviz 0.22.0 | fast mode: False


In [2]:
#| output: false
#| code-summary: "Show the loading of the data"
data = calce.load(Path(tempfile.gettempdir()) / "calce-battery-features.parquet", FEATURES, every=8)
train, held_out = calce.split(data, HELD_OUT)
scaler = StandardScaler().fit(train[FEATURES])  # standardised on the training cells only
CHEMISTRIES = list(dict.fromkeys(data["CellType"]))
COLOURS = hierarchy.palette(CHEMISTRIES)
CHEMISTRY_OF = data.drop_duplicates("cell").set_index("cell")["CellType"].to_dict()

summary = data.groupby(["CellType", "cell"]).agg(rows=("cycle", "size"), last_cycle=("cycle", "max"))
for chemistry, rows in summary.groupby("CellType"):
    print(f"{chemistry}: {len(rows)} cells, {rows['rows'].min()} to {rows['rows'].max()} kept cycles per cell, longest life {rows['last_cycle'].max()} cycles")
print(f"{len(data):,} cycles kept; {len(train):,} in the {train['cell'].nunique()} training cells; held out: {', '.join(HELD_OUT)}")
print(f"SoH above 100%: {(data['soh'] > 100).mean():.1%} of the kept cycles")

CS2: 6 cells, 78 to 108 kept cycles per cell, longest life 1056 cycles
CX2: 7 cells, 134 to 196 kept cycles per cell, longest life 2474 cycles
1,752 cycles kept; 1,456 in the 11 training cells; held out: CS2_38, CX2_38
SoH above 100%: 1.7% of the kept cycles


## Decision on Retiring a Cell

A model is useful to the operator only through the decision that it supports, so the decision is stated first. At an inspection the operator can retire the cell, at a cost $C$ that covers the replacement and the downtime, or leave it in service. A cell that is left in service while it is below the threshold of 80% causes a loss $L$, for example a failure of the string that it belongs to, and a cell that is above the threshold costs nothing. The two costs enter the decision only through their ratio,

$$
\alpha = \frac{C}{L},
$$

which is small when a failure is much more expensive than a replacement and close to one when retiring a cell costs almost as much as the loss that it prevents. The model supplies the probability $p$ that the state of health of the cell is below the threshold, which is read from the predictive distribution of the model. The operator expects to lose $pL$ by leaving the cell and $C$ by retiring it, so retiring is the cheaper choice when

$$
p > \alpha.
$$

@fig-decision draws the decision as three steps and a threshold on the probability. This is the cost-loss model of decision making under a probability forecast.[^murphy] The rule is the best one for an operator whose probabilities are reliable, and it shows what the operator needs from a model: probabilities that are neither too confident nor too vague. A model that states a probability near zero for a cell that is below the threshold, or near one for a cell that is above it, is paid for in replacements that were not needed and failures that were not avoided. The value of a model to an operator is then the share of the possible saving that it achieves, against an operator who has only the long-run rate of cells below the threshold, and it is 0 for that operator and 1 for one who knows the outcome.[^richardson] [Forecast Value, Part 1](/pages/blog/posts/2026/11/2026-11-24-forecast-value-events.html) derives the same rule and the same relative economic value for a limit on a distribution network, where the probability of an event comes from a quantile or Laplace forecaster. Here the probability comes from the posterior of a hierarchical model and the event is a single cell below 80%, so the two posts apply one decision to two sources of probability. The sections that follow build the model and test it as a predictor, and the last of them applies the rule to each strategy and shows what an operator would pay under each.

[^murphy]: A. H. Murphy, "The value of climatological, categorical and probabilistic forecasts in the cost-loss ratio situation," *Mon. Weather Rev.*, vol. 105, no. 7, pp. 803-816, 1977.

[^richardson]: D. S. Richardson, "Skill and relative economic value of the ECMWF ensemble prediction system," *Q. J. R. Meteorol. Soc.*, vol. 126, no. 563, pp. 649-667, 2000.

In [3]:
#| code-summary: "Show the code of the figure"
#| label: fig-decision
#| fig-cap: "The decision at an inspection: the model gives the probability p that the cell is below 80%, and the cell is retired when p exceeds the ratio α of the cost C of retiring to the loss L of a failure."
#| fig-alt: "Three steps, inspect, predict and compare, lead to a probability axis from 0 to 1. A dashed line at alpha, the cost C over the loss L, splits it. Below alpha the cell is left in service and risks the loss L. Above alpha it is retired at the cost C."
show(concepts.decision())

## Levels of Pooling

The question of how far a new cell resembles the cells already seen has three answers, which are called levels of pooling, and @fig-pooling draws them for the 13 cells of this study. Under **complete pooling** every cell shares one set of curve parameters, so a new cell receives the average curve of the fleet and its chemistry plays no part. Under **no pooling** each cell has parameters of its own and nothing is shared, so a cell is described by its own record only, and a cell without a record is described by the prior. **Partial pooling** lies between them: the parameters of a cell come from a distribution that belongs to its chemistry, and the parameters of a chemistry come from a distribution over the fleet. A cell with a long record is then free to move away from its chemistry, and a cell with a short or incomplete record is drawn towards it.

In [4]:
#| code-summary: "Show the code of the figure"
#| label: fig-pooling
#| fig-cap: "Three ways of tying the 13 cells together: one curve for the fleet, one curve for each cell, and a level for each chemistry between the fleet and its cells. Hollow rings mark the two held-out cells."
#| fig-alt: "Three diagrams of 13 battery icons, six blue cells of one chemistry and seven orange cells of another, the last of each chemistry ringed by a dashed line as held out. In complete pooling one fleet card with a curve icon is joined to every cell. In no pooling each cell has its own curve icon and the held-out cell has a question mark. In partial pooling the fleet is joined to a layered icon for each chemistry, and each chemistry to its cells, with a dashed line to its held-out cell."
#| fig-subcap: ["Complete pooling", "No pooling", "Partial pooling"]
#| layout-ncol: 3
for diagram in (concepts.complete, concepts.none, concepts.partial):
    show(diagram(CHEMISTRY_OF, HELD_OUT, COLOURS))

The three diagrams differ in what lies above the cells and so in what each strategy believes about a cell that it has not seen. The hollow rings are the two cells that are held out of the fits to test that belief, one of each chemistry. Complete pooling has no way to tell the new cells apart, no pooling has nothing to say about them except the prior, and partial pooling starts each of them from the level of its own chemistry. The choice matters most when records are short or incomplete, because the data then say little about a cell and the structure of the model decides what is believed. The sections that follow measure that on a real fleet, and the first of them describes the data.

## Thirteen Cells, Two Chemistries

The data are the cycle-by-cycle diagnostics of 13 lithium-ion cells from the CALCE laboratory, in the form prepared for Part 2.[^calce] Six cells are of chemistry CS2, with a nominal capacity of 1.10 Ah, and seven are of chemistry CX2, with 1.35 Ah. Every cell was cycled until it fell well below the end-of-life threshold of 80% state of health (SoH). The target is the SoH, the capacity divided by the nominal capacity, divided again by 1.1 so that it stays inside the open unit interval that a Beta likelihood requires. SoH exceeds 100% in 1.7% of the cycles, which the ceiling of 1.1 accommodates. The inputs are the four diagnostics of Part 2, standardised on the training cells only.

Every eighth cycle is kept, because neighbouring cycles are almost duplicates, which leaves 1,752 observations with between 78 and 196 per cell. Two cells, CS2_38 and CX2_38, the last of each chemistry, are held out to test the strategies on cells that the model has not seen, and the remaining 11 cells provide 1,456 observations for training. The inputs are measured in the same cycle as the capacity, which is what an operator has at an inspection, so the model estimates the state of health from diagnostics and does not forecast it.

[^calce]: CALCE Battery Research Group, University of Maryland, "Battery cycling data and diagnostic features." [Online]. Available: <https://ndownloader.figshare.com/files/59415941>

In [5]:
#| code-summary: "Show the code of the figure"
#| label: fig-curves
#| fig-cap: "State of health of the 13 cells against cycle. CX2 cells are orange and CS2 cells blue, dashed lines are the two cells held out for testing, and the dotted line is the end-of-life threshold of 80%."
#| fig-alt: "Line chart of state of health against cycle for 13 cells. The seven orange CX2 cells decline slowly and last up to 2,474 cycles. The six blue CS2 cells decline faster and fail between 737 and 1,056 cycles. A dotted line marks 80%."
show(lambda: hierarchy.curves(data, HELD_OUT, COLOURS))

The two chemistries differ markedly, as @fig-curves shows. The CX2 cells last up to 2,474 cycles and decline slowly, while the CS2 cells fail between 737 and 1,056 cycles. The records are not equally complete. CX2_16 begins at cycle 463 with a state of health of 94%, and CX2_33 begins at cycle 187, so for these two cells the early part of the curve is not observed. What the data say about their initial level is little, and a model that sees only these cells has to rely on its prior.

## The Hierarchical Model

The observation layer is a Beta regression, as in Part 2.[^beta] The state of health $y_{ij}$ of cell $j$ at its cycle $c_{ij}$ follows a Beta distribution with mean $\mu_{ij}$ and precision $\phi$,

$$
y_{ij} \sim \mathrm{Beta}\big(\mu_{ij}\phi,\ (1-\mu_{ij})\phi\big),
$$

and the mean is linked to the linear predictor by the inverse logit,

$$
\mu_{ij} = \mathrm{logit}^{-1}(\eta_{ij}).
$$

The linear predictor on the logit scale combines the ageing of the cell with the diagnostics $\mathbf{x}_{ij}$,

$$
\eta_{ij} = \alpha_j - A_k\, g_j(c_{ij}) + \mathbf{x}_{ij}^{\top}\boldsymbol{\beta},
$$

where ageing is a saturating exponential of the cycle,

$$
g_j(c) = 1 - e^{-\lambda_j c}.
$$

Here $\alpha_j$ is the initial level of cell $j$, $\lambda_j$ its rate, $A_k$ the amplitude of the chemistry $k$ to which it belongs, and the cycle is counted in thousands. The hierarchy ties the cell parameters to the chemistry. Each chemistry has an initial level $a_k$, a log rate $\ell_k$ and an amplitude $A_k$, drawn from distributions that are shared by the fleet, and a cell departs from its chemistry by an offset,

$$
\alpha_j = a_k + \sigma_{a,k}\, z_{a,j},
$$

$$
\log\lambda_j = \ell_k + \sigma_{\ell,k}\, z_{\ell,j},
$$

where the offsets $z_{a,j}$ and $z_{\ell,j}$ have a standard normal distribution.

The unit-normal offsets $z$ are sampled and scaled afterwards, which is known as a non-centred parameterisation. It matters when the spread between cells is small, because a model that samples $\alpha_j$ directly then has a funnel-shaped posterior that the sampler traverses poorly and reports as divergences.[^funnel]

The code defines one function for the three strategies, so that they differ in nothing else. For complete pooling the three parameters are single numbers, for no pooling each cell has its own with an independent prior, and for partial pooling they follow the hierarchy above. The hidden cell below builds the three models and fits them, as the section on sampling describes.

[^beta]: S. Ferrari and F. Cribari-Neto, "Beta regression for modelling rates and proportions," *J. Appl. Statist.*, vol. 31, no. 7, pp. 799-815, 2004.

[^funnel]: M. Betancourt and M. Girolami, "Hamiltonian Monte Carlo for hierarchical models," in *Current Trends in Bayesian Methodology with Applications*. CRC Press, 2015.

In [6]:
#| output: false
#| code-summary: "Show the model and its fits"
models = {pooling_level: hier.hierarchical_model(train, scaler, FEATURES, pooling_level) for pooling_level in hier.POOLING}
print({pooling_level: len(model.free_RVs) for pooling_level, model in models.items()})
fits = {
    pooling_level: hier.sample(models[pooling_level], CACHE / f"train-{pooling_level}.nc", seed=2026 + index, log_likelihood=True, **MAIN)
    for index, pooling_level in enumerate(hier.POOLING)
}

{'partial': 14, 'complete': 5, 'none': 5}


## Priors and Prior Checks

A prior states what the model believes before it sees any data, and the quickest way to understand one is to look at it. @fig-priors draws the prior of each quantity that the model learns. The priors are weakly informative, which means that they rule out values that no battery could show and leave the rest to the data.[^gelmanprior] The initial level has a normal prior centred on 2 with a standard deviation of 1 at the fleet level, which puts the state of health near 97% for a cell with no ageing term and no diagnostic effect. The log rate is centred on $\log 5$ per thousand cycles with a standard deviation of 0.5, so the saturating term of an ageing cell reaches half of its amplitude after about 140 cycles. The amplitude has a half-normal prior, which allows a small ageing term or a large one but not a negative one. The effect of each standardised diagnostic has a normal prior with a standard deviation of 0.2, so that a feature has to earn a large coefficient, and the standard deviations between cells and between chemistries have half-normal priors, as is usual for scale parameters. The precision $\phi$ of the Beta likelihood has a log-normal prior centred on 100, which corresponds to noise of about 5 points of state of health at mid-life. @tbl-priors lists them all.

[^gelmanprior]: A. Gelman, "Prior distributions for variance parameters in hierarchical models," *Bayesian Anal.*, vol. 1, no. 3, pp. 515-534, 2006.

In [7]:
#| label: tbl-priors
#| tbl-cap: "Priors of the hierarchical Beta regression. The fleet level and the spreads are shared, and each cell draws its parameters from its chemistry through the cell spreads."
display(tables.themed_gt(priors.hierarchical()))

Parameter,Prior,Role
Fleet level of α,"Normal(2, 1)",Mean initial level across chemistries
Spread of α between chemistries,HalfNormal(1),How far chemistries differ in initial level
Fleet level of ln λ,"Normal(ln 5, 0.5²)",Mean log rate across chemistries
Spread of ln λ between chemistries,HalfNormal(0.5),How far chemistries differ in rate
Amplitude scale,HalfNormal(1),Scale of the chemistry amplitudes
Amplitude A of a chemistry,HalfNormal(scale),Size of the decay of a chemistry
"Spread of α, ln λ between cells",HalfNormal(0.5),How far a cell departs from its chemistry
Effects β,"Normal(0, 0.2²)",Effect of each standardised diagnostic
Precision φ,"LogNormal(ln 100, 1)",Noise of the Beta likelihood


In [8]:
#| output: false
#| code-summary: "Show the draws of the prior"
with models["partial"]:
    prior = pm.sample_prior_predictive(draws=500, random_seed=1)
    prior_many = pm.sample_prior_predictive(draws=3000, random_seed=2)
prior_soh = prior.prior_predictive["y"].to_numpy().ravel() * hier.CEILING * 100
drawn = np.random.default_rng(1).choice(prior_soh, size=min(20000, prior_soh.size), replace=False)
update = hier.prior_posterior(prior_many, fits["partial"])
QUANTITIES = list(hier.PRIOR_LEVELS.values()) + ["Precision (log10)"]

Sampling: [alpha_chem, amplitude_chem, amplitude_scale, beta, loglambda_chem, mu_alpha, mu_lambda, phi, sigma_alpha, sigma_cell_alpha, sigma_cell_lambda, sigma_lambda, y, z_alpha, z_lambda]


Sampling: [alpha_chem, amplitude_chem, amplitude_scale, beta, loglambda_chem, mu_alpha, mu_lambda, phi, sigma_alpha, sigma_cell_alpha, sigma_cell_lambda, sigma_lambda, y, z_alpha, z_lambda]


In [9]:
#| code-summary: "Show the code of the figure"
#| label: fig-priors
#| fig-cap: "Prior densities of the initial level, the log rate and the amplitude of a chemistry, and of the precision of the Beta likelihood on a logarithmic scale, each drawn before the model sees any data."
#| fig-alt: "Four density curves in slate. The initial level is a bell centred on 2, the log rate a bell centred near 1.6, the amplitude a skewed curve that falls from zero, and the precision a bell centred on 2 on a base-10 logarithmic scale."
#| fig-subcap: ["Initial level (logit)", "Log rate (per 1,000 cycles)", "Amplitude (logit units)", "Precision (log10)"]
#| layout-ncol: 2
for quantity in QUANTITIES:
    show(lambda quantity=quantity: hierarchy.priors(update[update["source"] == "Prior"], quantity, COLOURS))

The four priors are broad on purpose. The initial level and the log rate cover every ageing curve that a lithium-ion cell could plausibly follow, the amplitude allows any size of ageing term, and the precision prior expects the noise of a fair sensor, not of a cell that one diagnostic describes almost exactly. A prior that is broad in this way commits the model to very little, and the posterior shows how much the data then changed it. What the four priors imply together for the state of health can be checked before any data are used, by drawing parameters from the priors and simulating the cells that they describe. @fig-prior compares the state of health of that prior predictive distribution with the observed values of the training cells.

In [10]:
#| code-summary: "Show the code of the figure"
#| label: fig-prior
#| fig-cap: "Observed state of health of the training cells in blue and the prior predictive distribution of the model in slate, as densities against the state of health in percent."
#| fig-alt: "Two density curves of state of health in percent. The observed values in blue peak between 80% and 90%. The prior predictive values in slate are much wider, from near 0% to above 100%."
show(lambda: hierarchy.prior(train["soh"].to_numpy(), drawn))

The prior has its 5th, 50th and 95th percentiles at 23.6%, 91.1% and 109.1%, against 50.8%, 81.9% and 97.7% in the data. It is wider than the data at both ends and its median lies nine points above the observed one, and some of its mass exceeds 100%, which the data also do in 1.7% of the cycles. A weak prior should rule out the impossible and leave the rest to the data, and this one does.

## Posterior Sampling

The posteriors are sampled with the No-U-Turn sampler.[^nuts] Every fit runs four chains of 500 draws after 750 tuning draws, with a target acceptance rate of 0.90, and this covers the three models of the training cells, the eight refits of the test below the two refits of the check of the prior and the four refits of its scores. Sampling is the slow part. The eight refits took 34 minutes on a laptop, the hierarchical ones taking about 8 minutes each, and the four refits of the scores under other priors took 27 minutes, so the notebook stores each fit in a `cache` folder and reads it back on a repeat run.

[^nuts]: M. D. Hoffman and A. Gelman, "The No-U-Turn Sampler: Adaptively setting path lengths in Hamiltonian Monte Carlo," *J. Mach. Learn. Res.*, vol. 15, pp. 1593-1623, 2014.

### Convergence of the Chains

None of the 12 fits of the post, the three on the training cells, the eight refits and the control, has a divergent transition. For the three fits on the training cells the worst $\hat{R}$, the disagreement between chains, is 1.000 for partial pooling, 1.010 for complete pooling and 1.020 for no pooling, against a common guide of 1.01.[^rhat] The smallest bulk effective sample sizes are 820, 1,008 and 392, so the no-pooling model is the least well behaved. Across all 12 fits the worst $\hat{R}$ is 1.02 and the smallest bulk effective sample size is 392. The six refits of the checks of the prior, which are described below, have a worst $\hat{R}$ of 1.01 and a smallest bulk effective sample size of 572. Four divergent transitions occur in their 12,000 draws, two with the wide prior with no cycles seen, one in each refit after 300 cycles, and the estimates of those fits agree with those of the fits without one.

[^rhat]: A. Vehtari, A. Gelman, D. Simpson, B. Carpenter, and P.-C. Bürkner, "Rank-normalization, folding, and localization: An improved $\hat{R}$ for assessing convergence of MCMC," *Bayesian Anal.*, vol. 16, no. 2, pp. 667-718, 2021.

In [11]:
#| output: false
#| code-summary: "Show the checks of the sampler"
for pooling_level, idata in fits.items():
    report = hier.convergence(idata)
    print(f"{pooling_level}: {idata.posterior.sizes['chain']} chains of {idata.posterior.sizes['draw']} draws, {report['divergences']} divergences, max R-hat {report['r_hat']:.3f}, min ESS bulk {report['ess_bulk']:.0f}, min ESS tail {report['ess_tail']:.0f}")

partial: 4 chains of 500 draws, 0 divergences, max R-hat 1.000, min ESS bulk 820, min ESS tail 717
complete: 4 chains of 500 draws, 0 divergences, max R-hat 1.010, min ESS bulk 1008, min ESS tail 873
none: 4 chains of 500 draws, 0 divergences, max R-hat 1.020, min ESS bulk 392, min ESS tail 857


## From Prior to Posterior

The data turn each prior into a posterior, and the distance between the two says how much the data taught the model. @fig-update draws the log rate of each chemistry and the precision, each against its prior. Where a posterior is narrow and far from the prior, the data decided the answer, and where it stays close to the prior, the prior did. The first quantity decides how quickly the ageing term of a chemistry saturates, and the second decides how closely the model believes that its predictions follow the observations, so together they show where the data were informative and where they were not.

In [12]:
#| code-summary: "Show the code of the figure"
#| label: fig-update
#| fig-cap: "Prior and posterior of the log rate of each chemistry, and of the precision on a logarithmic scale. The slate curve is the prior, blue is CS2 and the posterior of the precision, and orange is CX2."
#| fig-alt: "Left, the prior of the log rate in slate, broad and centred near 1.6, with a narrow blue posterior for CS2 near minus 0.3 and a narrow orange posterior for CX2 near 0.7. Right, a flat slate prior of the precision and a very narrow blue posterior at about 3.3 on the base-10 logarithmic scale, far to its right."
#| fig-subcap: ["Log rate (per 1,000 cycles)", "Precision (log10)"]
#| layout-ncol: 2
for quantity in ("Log rate (per 1,000 cycles)", "Precision (log10)"):
    show(lambda quantity=quantity: hierarchy.priors(update, quantity, COLOURS))

The log rate of the two chemistries moved well away from the prior, which is centred on $\log 5 = 1.61$. The posterior mean is $-0.31$ for CS2 and 0.73 for CX2, so the data separated the chemistries and placed both well below the prior. The posterior of CS2 is the broader of the two, with a 90% interval from $-0.78$ to 0.30 against 0.44 to 1.05 for CX2, because fewer cells inform it and they differ from one another. The precision moved furthest. Its posterior mean is 1,810 against a prior centred on 100, and the prior puts only 0.2% of its mass above that value, so the figure shows it on a logarithmic scale, where the posterior is a narrow spike at 3.26 far to the right of a prior centred on 2. The data are therefore far more precise than the prior expected, because one diagnostic nearly determines the state of health. This is the one prior of the model that the data contradict, and a later section tests whether it matters.

## Posterior of the Chemistries

The chemistry level of the hierarchical model is where the model says something that a single curve cannot. @fig-chemistry shows the posterior of the three parameters for each chemistry, one parameter in each panel. They differ in all three. The initial level is 1.19 for CS2 (90% interval 1.08 to 1.32) and 1.98 for CX2 (1.72 to 2.22) on the logit scale. The amplitude of the saturating term is 2.76 for CS2 (2.33 to 3.29) and 1.03 for CX2 (1.01 to 1.06). The log rate is $-0.31$ for CS2 (from $-0.78$ to 0.30) and 0.73 for CX2 (from 0.44 to 1.05), and the posterior probability that the rate of CX2 exceeds that of CS2 is 0.992.

In [13]:
#| code-summary: "Show the code of the figure"
#| label: fig-chemistry
#| fig-cap: "Posterior density of the three chemistry parameters, for CS2 in blue and CX2 in orange. The further apart the curves, the more surely the chemistries differ."
#| fig-alt: "Three density charts, each with a blue curve for CS2 and an orange curve for CX2. CX2 has the higher initial level and rate and the smaller amplitude, and the curves overlap little."
#| fig-subcap: ["Initial level (logit)", "Log rate (per 1,000 cycles)", "Amplitude (logit units)"]
#| layout-ncol: 3
draws = hier.chemistry_draws(fits["partial"])
for quantity in hier.PRIOR_LEVELS.values():
    show(lambda quantity=quantity: hierarchy.chemistry(draws, COLOURS, quantity))

Read together, the three describe the curves of @fig-curves. The CX2 term is small and saturates early, with a half-time of about 330 cycles, so the long, slow decline of those cells is carried by the diagnostics. The CS2 term is large and slow, with a half-time of about 950 cycles, so those cells keep losing capacity over their whole life. The difference between the chemistries is larger than the difference between cells within one. The standard deviation between cells of the log rate is 0.52 in CS2 and 0.47 in CX2, against a gap of 1.04 between the chemistries, and for the initial level it is 0.14 and 0.34, against a gap of 0.79. Only two chemistries inform these standard deviations, so they are set partly by their priors and are not interpreted further.

Of the four diagnostics, only the area under the discharge voltage has an effect that differs from zero beyond doubt, at 0.362 with a posterior standard deviation of 0.016, and the other three lie within two standard deviations of zero. The precision $\phi$ is about 1,800, which corresponds to a noise standard deviation of about one point of state of health. That is far in the upper tail of its prior, which puts 0.2% of its mass above 1,800, and it says that one diagnostic nearly determines the state of health, which has consequences for the tests that follow.

### Cell Shrinkage

The hierarchy also changes what is said about a single cell. @fig-shrinkage sets the estimate of each training cell without pooling, hollow, beside the estimate with partial pooling, filled, with the posterior mean of the chemistry as a dashed line.

In [14]:
#| code-summary: "Show the code of the figure"
#| label: fig-shrinkage
#| fig-cap: "Estimates of a parameter of each training cell without pooling (hollow) and with partial pooling (filled), with the posterior mean of each chemistry as a dashed line."
#| fig-alt: "Two dot charts over the 11 training cells. Hollow dots are estimates without pooling and filled dots with partial pooling, joined by a segment, with dashed lines at the mean of each chemistry. The initial level of CX2_16 moves most towards the line of its chemistry."
#| fig-subcap: ["Initial level (logit)", "Log rate (per 1,000 cycles)"]
#| layout-ncol: 2
chemistry_of = train.drop_duplicates("cell").set_index("cell")["CellType"]
cells = [cell for chemistry in CHEMISTRIES for cell in chemistry_of[chemistry_of == chemistry].index]
estimates, means = hier.shrinkage_frames(fits["none"], fits["partial"], chemistry_of, cells)
for parameter in ("Initial level (logit)", "Log rate (per 1,000 cycles)"):
    show(lambda parameter=parameter: hierarchy.shrinkage(estimates, means, COLOURS, cells, parameter))

The largest change is for the cell with the least complete record. The initial level of CX2_16, whose record begins at cycle 463, moves from 3.54 without pooling to 1.96 with it, close to the 1.98 of its chemistry, and CX2_33, which begins at cycle 187, moves from 2.09 to 1.74. The other cells change by 0.12 or less. Without pooling, the initial level of a cell with no early record is set by the prior and by observations far from cycle zero, and the hierarchy replaces that with what its chemistry shows.

The log rate changes more widely, by up to 0.94 in CS2_35 and 0.92 in CS2_33, although those cells have complete records. On a curve that has not yet saturated, the rate and the amplitude trade off against each other, so a cell's rate is poorly determined on its own, and the shared amplitude of its chemistry fixes part of that trade. This reading is consistent with the figure but was not tested separately. Averaged over the training cells, an estimate differs by 0.24 logit units for the initial level and by 0.40 for the log rate between the two strategies.

## Sensitivity to the Prior

The posterior of the precision lies far from its prior, and a prior that is far from the data can decide the answer when the data are few. With 1,456 observations that is unlikely, but the question can be answered by computation and the limits of the first version of this study left it open. The model is therefore refitted with a narrow prior for the precision, a log-normal distribution centred on 100 whose logarithm has a standard deviation of 0.3, and with a wide prior centred on 1,000 whose logarithm has a standard deviation of 1.5, against the default of 1.0 centred on 100. @fig-sensitivity draws the posterior of the log rate of each chemistry under the three priors, with the default fit as the heavier line.

In [15]:
#| output: false
#| code-summary: "Show the refits under other priors"
sensitivity_fits = {}
for index, (setting, phi_prior) in enumerate(PHI_PRIORS.items()):
    if setting.startswith("Default"):
        sensitivity_fits[setting] = fits["partial"]  # the fit of the post, not a second copy of it
        continue
    model = hier.hierarchical_model(train, scaler, FEATURES, "partial", phi_prior=phi_prior)
    sensitivity_fits[setting] = hier.sample(model, CACHE / f"prior-phi-{setting.split(',')[0].lower()}.nc", seed=4000 + index, **REFIT)
sensitivity = hier.prior_sensitivity(sensitivity_fits)
sensitivity_draws = hier.sensitivity_draws(sensitivity_fits)
for setting, idata in sensitivity_fits.items():
    report = hier.convergence(idata)
    print(f"{setting}: {report['divergences']} divergences, max R-hat {report['r_hat']:.3f}, min ESS bulk {report['ess_bulk']:.0f}")

Narrow: 0 divergences, max R-hat 1.010, min ESS bulk 769
Default: 0 divergences, max R-hat 1.000, min ESS bulk 820


Wide: 2 divergences, max R-hat 1.010, min ESS bulk 815


In [16]:
#| code-summary: "Show the code of the figure"
#| label: fig-sensitivity
#| fig-cap: "Posterior density of the log rate of each chemistry under a narrow, a default and a wide prior for the precision of the Beta likelihood."
#| fig-alt: "Two panels of three overlaid density curves, one for CS2 centred near minus 0.3 and one for CX2 centred near 0.7. The curves for the narrow, default and wide priors lie almost on top of one another."
#| fig-subcap: ["CS2", "CX2"]
#| layout-ncol: 2
for chemistry in CHEMISTRIES:
    show(lambda chemistry=chemistry: hierarchy.sensitivity(sensitivity_draws, chemistry))

The three curves of each chemistry lie almost on top of one another. The posterior mean of the log rate of CS2 is $-0.36$, $-0.32$ and $-0.33$ under the narrow, the default and the wide prior, and that of CX2 is 0.73, 0.73 and 0.74. The probability that the rate of CX2 exceeds that of CS2 is 0.996, 0.992 and 0.990. The initial level and the amplitude of each chemistry agree to within 0.02 across the three priors. The posterior mean of the precision is 1,739, 1,810 and 1,820, which says that the data move the precision from a prior centred on 100 to about 1,800 even when the prior is tight. The conclusions about the chemistries therefore do not depend on the prior of the precision, and the tight prior of the default model does not decide them. The same parameter sets the width of every predictive interval, so the last section repeats the check on the scores.

## Fit on Training Cells

Before the model is tested on cells that it has not seen, it is checked on the cells that it has. Leave-one-out cross-validation scores a model by the log density that it gives to an observation when that observation is left out of the fit, and the average of these scores is the expected log predictive density.[^loo] Pareto-smoothed importance sampling approximates the scores from a single fit, and the difference between two models comes with a standard error. The log density is a proper scoring rule, so the model with the larger value is the one whose predictive distribution lies closer to the process that generated the data. Its scale is that of the Beta density of the state of health as a fraction, so only differences between models have a meaning. @fig-loo shows, for each of the 11 training cells, how much partial pooling and complete pooling gain over no pooling, and the bars of one model add up to its difference in expected log predictive density. Partial pooling is behind no pooling by 124.6 (standard error 14.5), which is 8.6 standard errors, and complete pooling is behind by 2,242 (standard error 76). Partial pooling is behind in 10 of the 11 cells and its largest loss is 31.3 for CX2_16, while complete pooling is behind in every cell. The effective number of parameters is 63.6 for no pooling, 45.9 for partial pooling and 10.5 for complete pooling. One observation of the no-pooling model and one of the partial-pooling model have a Pareto $k$ above 0.7, so those two totals are a little less certain than the standard errors say.

[^loo]: A. Vehtari, A. Gelman and J. Gabry, "Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC," *Stat. Comput.*, vol. 27, pp. 1413-1432, 2017.

In [17]:
#| output: false
#| code-summary: "Show the leave-one-out scores"
labels = {"none": "No pooling", "complete": "Complete pooling", "partial": "Partial pooling"}
named = {labels[pooling_level]: fits[pooling_level] for pooling_level in hier.POOLING}
comparison = az.compare(named, ic="loo")
print(comparison[["elpd_loo", "p_loo", "elpd_diff", "dse"]].round(1).to_string())
by_cell = hier.loo_by_cell(named, train["cell"].to_numpy())
wide = by_cell.pivot(index="cell", columns="method", values="elpd")
gain = wide.sub(wide["No pooling"], axis=0)
print("cells where partial pooling beats no pooling:", int((gain["Partial pooling"] > 0).sum()), "of", len(gain))
print("largest loss of partial pooling:", gain["Partial pooling"].idxmin(), round(gain["Partial pooling"].min(), 1))
print("cells where complete pooling beats no pooling:", int((gain["Complete pooling"] > 0).sum()))
print("observations with Pareto k above 0.7:", by_cell.groupby("method")["bad"].sum().to_dict())

                  elpd_loo  p_loo  elpd_diff   dse
No pooling          4760.0   63.6        0.0   0.0
Partial pooling     4635.5   45.9      124.6  14.5
Complete pooling    2517.8   10.5     2242.3  76.0


cells where partial pooling beats no pooling: 1 of 11
largest loss of partial pooling: CX2_16 -31.3
cells where complete pooling beats no pooling: 0
observations with Pareto k above 0.7: {'Complete pooling': 0, 'No pooling': 1, 'Partial pooling': 1}


In [18]:
#| code-summary: "Show the code of the figure"
#| label: fig-loo
#| fig-cap: "Gain in expected log density of partial pooling and of complete pooling over no pooling, for each cell, from leave-one-out cross-validation. Bars to the left of zero are cells that the model predicts worse."
#| fig-alt: "Two bar charts with one bar for each of the 11 training cells, blue for CS2 and orange for CX2. Partial pooling is within a few units of no pooling for most cells. Complete pooling loses many units for every cell."
#| fig-subcap: ["Partial pooling", "Complete pooling"]
#| layout-ncol: 2
for method in ("Partial pooling", "Complete pooling"):
    show(lambda method=method: hierarchy.loo(by_cell, COLOURS, CHEMISTRY_OF, method, "No pooling"))

Cell-specific parameters matter a great deal for cells that are in the data, and the likelihood is so sharp that an offset of two or three points of state of health is worth many units of log density, which is why every cell loses under complete pooling. The same parameters are not available for a new cell, so this ranking does not carry over to one. Leave-one-out here leaves out single cycles, and the neighbouring cycles of the same cell still fix its curve, so it measures how well a model interpolates within cells it has seen and not how well it predicts a cell that it has not. The test of the next section leaves out whole cells and answers that question; a leave-one-cell-out version of this figure would need one refit for each of the 11 cells and each model.

## Test on Unseen Cells

The test gives each strategy a new cell and asks for its state of health over the rest of its life. A new cell has no parameters in the fitted model, so its curve cannot be read from the posterior. The parameters are drawn instead from what each strategy says about a new cell. Under partial pooling the cell takes the chemistry level of its type and adds the offsets $\sigma_{a,k} z_a$ and $\sigma_{\ell,k} z_\ell$ with fresh unit-normal draws, which carries the cell-to-cell spread into the prediction. Under complete pooling it takes the common parameters, and under no pooling it takes its prior, since no pooled information exists. In each case the diagnostics of the cell go through the fitted effects, and the noise of the Beta likelihood is added to give predictive draws.

The strategies are first given no cycle of the new cell, and then its first 100 or 300 cycles, which are added to the training data before the models are refitted. All settings are scored on the same cycles, those after cycle 350 of each held-out cell, 222 observations in all, so that the scores differ only because of what was seen. Each setting is scored with the error of the median in points of state of health, the coverage of the 90% interval and the interval score, which is the width of the interval plus $2/\alpha$ times the distance by which an observation falls outside it, with $\alpha = 0.1$ and averaged over the observations. The interval score is a proper score, so it cannot be lowered by an interval that is too wide or too narrow for the truth.[^gneiting] A fourth model, complete pooling without the diagnostics, serves as a control: if the diagnostics carry the information, removing them should hurt. The same predictive draws give, for each scored cycle, the probability that the state of health is below 80%, which the last section turns into a retirement rule.

[^gneiting]: T. Gneiting and A. E. Raftery, "Strictly proper scoring rules, prediction, and estimation," *J. Amer. Statist. Assoc.*, vol. 102, no. 477, pp. 359-378, 2007.

In [19]:
#| output: false
#| code-summary: "Show the test on unseen cells"
result = evaluate.cold_start(train, held_out, scaler, FEATURES, SEEN, GAP, MAIN, REFIT, CACHE)
scores, diagnostics, curves, windows = result

### Prediction without Cycles

@fig-predict shows the predictions of complete pooling and of partial pooling for CS2_38 and CX2_38 when the model has seen none of their cycles, with the median as a line, the 90% interval as a band and the observations as points. The prediction of no pooling is its prior, which is too wide to draw usefully, with an interval of 73.6 points.

In [20]:
#| code-summary: "Show the code of the figure"
#| label: fig-predict
#| fig-cap: "Predicted state of health of the two held-out cells before the model has seen any of their cycles. The line is the median, the band the 90% interval and the points the observations."
#| fig-alt: "Two charts of state of health against cycle for CS2_38 and CX2_38. Under complete pooling the bands follow the observations closely. Under partial pooling the bands are wider and the CS2_38 median lies below its observations late in life."
#| fig-subcap: ["Complete pooling", "Partial pooling"]
#| layout-ncol: 2
for strategy in ("complete", "partial"):
    shown = curves[(curves["strategy"] == strategy) & (curves["cycles_seen"] == 0)]
    show(lambda shown=shown: hierarchy.bands(shown, COLOURS, 0, THRESHOLD, size=(440, 300)))

Complete pooling predicts both cells well. Its median is off by 3.0 points of state of health on average, its interval is 17.1 points wide, and the interval contains 95.5% of the observations. Partial pooling is less accurate and wider, with a median error of 4.7 points, an interval 31.5 points wide and a coverage of 100%. No pooling returns its prior, with a median error of 6.6 points and an interval 73.6 points wide. The control without diagnostics has a median error of 6.8 points and an interval score of 60.7, against 18.6 for complete pooling, so the diagnostics cut the error by more than half and the interval score to about a third.

The ranking is the opposite of what the hierarchy was built for, and it has a reason. One diagnostic nearly determines the state of health, so the average curve with the right diagnostics is already close. A hierarchical prediction adds the spread between cells on top of that, which makes the interval wide. The coverage of 100% against a nominal 90% means that the interval is conservative, not that it is accurate.

### Over-Confidence after More Cycles

The prediction changes when the model has seen the first cycles of a new cell, which are added to the training data before the model is refitted. @fig-bands shows partial pooling for the same two cells after the first 100 and after the first 300 cycles. The band is a prediction for the rest of the life of the cell, so about nine of ten observations should fall inside it.

In [21]:
#| code-summary: "Show the code of the figure"
#| label: fig-bands
#| fig-cap: "Predicted state of health of the two held-out cells under partial pooling after the model has seen the first 100 and the first 300 cycles of each. The dashed line marks the cycles seen, and the key of the first panel serves both."
#| fig-alt: "Two charts of state of health against cycle; the first has a key for the median, the 90% interval, the observations, the end-of-life line and the cycles seen. After 100 cycles the band of CS2_38 is wide and holds the observations. After 300 cycles the bands are narrow and the observations of CS2_38 lie above its band."
#| fig-subcap: ["First 100 cycles seen", "First 300 cycles seen"]
#| layout-ncol: 2
for seen in (100, 300):
    shown = curves[(curves["strategy"] == "partial") & (curves["cycles_seen"] == seen)]
    show(lambda shown=shown, seen=seen: hierarchy.bands(shown, COLOURS, seen, THRESHOLD, size=(440, 300), key=seen == 100))

After 100 cycles the band of CS2_38 is wide and holds its observations, and none of them lies outside it. After 300 cycles the band is narrow and every scored observation of CS2_38 lies above it, because the model has learnt the early decline of the cell and extrapolates a steeper fall than the cell shows. The median is 8.4 points of state of health too low on average. The band of CX2_38 narrows as well, and 22% of its observations lie above it, against 6% after 100 cycles. The picture is that of an over-confident prediction: the model is surer after 300 cycles than the cell allows. The scores of the next section put numbers on it, with a coverage of the 90% intervals of 94.6% after 100 cycles and 52.3% after 300.

In [22]:
#| code-summary: "Show the code of the figure"
#| label: fig-change
#| fig-cap: "Interval score of each strategy as the change in percent from complete pooling. Orange bars are worse than the reference, and partial pooling is blue where it is better."
#| fig-alt: "Three bar charts of the change in interval score from complete pooling for partial pooling, no pooling and the control, for a new cell that has shown no cycles, 100 cycles and 300 cycles."
#| fig-subcap: ["No cycles seen", "First 100 cycles seen", "First 300 cycles seen"]
#| layout-ncol: 3
ORDER = ["partial", "none", "control"]
shown = evaluate.change_by_strategy(scores, "interval", "complete", ORDER, hierarchy.STRATEGIES)
names = [hierarchy.STRATEGIES[name] for name in ORDER]
for panel in sorted(shown["panel"].unique()):
    show(lambda panel=panel: change.change(shown[shown["panel"] == panel], "Complete pooling", names, subject="Partial pooling", ncol=1, scale=shown))

@fig-change expresses the interval score of each strategy as the change from complete pooling, and the text gives the coverage of the 90% intervals, which should be near 90%. Complete pooling is the reference because it has no cell-specific parameters to update: its error stays between 2.99 and 3.04 points, its coverage near 95% and its interval score at about 18.6 in every setting. The control without diagnostics is worse than the reference by more than 220% in every panel, which confirms that the diagnostics carry the information.

After 100 cycles, partial pooling has an error of 3.2 points and a coverage of 94.6%, so it matches the calibration of complete pooling with an interval of 9.9 points against 17.0. Its interval score is 10.3 against 18.7, 45% below the reference, the one result in the test in which the hierarchy beats the single curve by a wide margin. No pooling has the smallest error, 1.6 points, but a coverage of 100% and an interval 22 points wide, so its interval score is 18% above the reference. After 300 cycles the picture reverses. Partial pooling and no pooling have intervals of about 5 points and cover only 52% and 29% of the observations, so they are over-confident, and their interval scores are 133% and 109% above the reference. The Beta likelihood treats the residuals of successive cycles as independent noise of about one point, whereas the real residuals of one cell are strongly correlated along its life, so more data made the calibration worse. That is a sign that the observation model is too simple and not that partial pooling failed. Two held-out cells are a small test, and these differences should be read as observations and not as estimates of a general effect.

## Retirement under Each Strategy

The decision of the first section can now be scored, in the same way as in [Forecast Value, Part 1](/pages/blog/posts/2026/11/2026-11-24-forecast-value-events.html), with the posterior probability of the strategy in place of the probability from a quantile forecaster. The scored inspections are the 222 kept cycles of the two held-out cells after cycle 350, 72 of CS2_38 and 150 of CX2_38, and 144 of them, 64.9%, are below 80%. At each inspection the operator retires the cell when the probability $p$ of the strategy exceeds the ratio $\alpha$, and the expense is $\alpha$ for a retired cell, 1 for a cell below the threshold that is left, and 0 otherwise, in units of the loss. Retiring every cell costs $\alpha$ and retiring none costs 0.649, so a strategy is worth having only where its expense is lower than the cheaper of the two, which is what the relative economic value measures. @fig-value draws the value of each strategy against the ratio, for a new cell that has shown no cycles, the first 100 and the first 300, up to a ratio of 0.9, because the possible saving that the value divides by tends to zero as the ratio approaches 1 and the value becomes unstable. A curve is read at one ratio: its height is the share of the possible saving that the strategy achieves, a curve at zero is no better than the operator without a model, and a curve below zero is worse. The orange curve is the control, which has no diagnostics and therefore shows what the diagnostics add.

In [23]:
#| output: false
#| code-summary: "Show the scoring of the decision"
values = decision.value_frame(windows, RATIOS, THRESHOLD)
events = windows[(windows["strategy"] == "complete") & (windows["cycles_seen"] == 0)]
EVENT_RATE = float((events["soh"] < THRESHOLD).mean())
print(f"scored inspections: {len(events)}; below {THRESHOLD:.0f}%: {int((events['soh'] < THRESHOLD).sum())} ({EVENT_RATE:.1%})")

scored inspections: 222; below 80%: 144 (64.9%)


In [24]:
#| code-summary: "Show the code of the figure"
#| label: fig-value
#| fig-cap: "Relative economic value of each strategy for retiring a cell, against the cost-loss ratio; values below -1 are drawn at -1. Zero is an operator who knows only the long-run rate, one an operator who knows the outcome. Orange is the control."
#| fig-alt: "Three line charts of relative economic value against cost-loss ratio for partial pooling in blue, complete pooling and no pooling in slate, and the control without diagnostics in orange, for a new cell that has shown no cycles, 100 cycles and 300 cycles."
#| fig-subcap: ["No cycles seen", "First 100 cycles seen", "First 300 cycles seen"]
#| layout-ncol: 3
for seen in SEEN:
    show(lambda seen=seen: hierarchy.value(values[values["cycles_seen"] == seen]))

No strategy is best at every ratio, and the ratio decides what the extra information is worth. With no cycles seen, complete pooling is the best of the three strategies with diagnostics at every ratio that the text quotes: at $\alpha = 0.2$, a failure that costs five times a retirement, its value is 0.37, while partial pooling has a value of zero. Its interval of 31.5 points puts $p$ above 0.2 for every cycle, so it retires every cell, which an operator could do without a model. After 100 cycles the cell-specific strategies pay. At $\alpha = 0.1$ the value of partial pooling is 0.33 against 0.12 for complete pooling, so a manufacturer that tests a new cell for 100 cycles and retires cells early saves about a third of what a perfect forecast would save. Partial pooling leads the three strategies only up to that ratio, however. Beyond it no pooling has the higher value over most of the range, 0.50 against 0.45 at $\alpha = 0.2$ and 0.64 against 0.55 at $\alpha = 0.3$, so after 100 cycles the hierarchy is the better choice only for an operator whose failures cost ten times a retirement or more. After 300 cycles partial pooling is the best of the three for every ratio up to 0.36, with a value of 0.46 at $\alpha = 0.1$ against 0.12 for complete pooling, even though its intervals are over-confident. The two facts agree: the model has learnt that CS2_38 declines early and over-estimates the decline, and a probability that is too high costs little when the cost of a retirement is small. At $\alpha = 0.5$, where retiring costs half of a failure, the picture reverses. Partial pooling has a value of 0.62 after 300 cycles, complete pooling 0.67 and no pooling 0.76, and the control without diagnostics, whose vague intervals keep $p$ near the middle, has the highest value of all, 0.85. An operator with balanced costs gains little from the sharper probabilities and loses by their over-confidence. The model matters most to the operator who must avoid failures, and for that operator the hierarchy is the better model once the new cell has shown 300 cycles, and after 100 only at the lowest ratios. This evidence rests on two cells.

In [25]:
#| output: false
#| code-summary: "Show the scores under other priors for the precision"
others = {setting: phi_prior for setting, phi_prior in PHI_PRIORS.items() if not setting.startswith("Default")}
phi_result = evaluate.phi_sensitivity(
    train, held_out, scaler, FEATURES, SEEN, GAP, REFIT, CACHE, others,
    {setting: sensitivity_fits[setting] for setting in others},
)
default = {"strategy": "Default"}
partial = scores[scores["strategy"] == "partial"].assign(strategy="Default")
phi_scores = pd.concat([partial, phi_result.scores], ignore_index=True)
phi_windows = pd.concat([windows[windows["strategy"] == "partial"].assign(strategy="Default"), phi_result.windows], ignore_index=True)
phi_value = decision.value_frame(phi_windows, RATIOS, THRESHOLD)
table = phi_scores.pivot(index="cycles_seen", columns="strategy", values=["coverage", "interval"]).round(3)
print(table.to_string())
for ratio in (0.1, 0.2, 0.5):
    rows = phi_value[np.isclose(phi_value["ratio"], ratio)].pivot(index="cycles_seen", columns="strategy", values="value").round(3)
    print(f"value at ratio {ratio}:"); print(rows.to_string())
print(phi_result.diagnostics.round(3).to_string(index=False))

            coverage               interval                
strategy     Default Narrow   Wide  Default  Narrow    Wide
cycles_seen                                                
0              1.000  1.000  1.000   31.525  34.522  34.086
100            0.946  0.959  0.955   10.343  10.118  10.018
300            0.523  0.545  0.550   43.271  42.301  43.404
value at ratio 0.1:
strategy     Default  Narrow   Wide
cycles_seen                        
0              0.000   0.000  0.000
100            0.333   0.333  0.333
300            0.462   0.462  0.462
value at ratio 0.2:
strategy     Default  Narrow   Wide
cycles_seen                        
0              0.000   0.000  0.000
100            0.449   0.462  0.487
300            0.526   0.538  0.526
value at ratio 0.5:
strategy     Default  Narrow   Wide
cycles_seen                        
0              0.628   0.615  0.628
100            0.641   0.628  0.654
300            0.615   0.615  0.615
strategy  cycles_seen  divergences  r_ha

### Scores under Other Priors

The precision also sets the width of every predictive interval, so the check of the prior is repeated on the scores. Partial pooling is refitted under the narrow and under the wide prior of the precision, with no cycles of the new cell seen and with the first 100 and the first 300, and scored on the same 222 inspections. The coverage of the 90% intervals is 100% in all three settings with no cycles seen, between 94.6% and 95.9% after 100 cycles and between 52.3% and 55.0% after 300, so the over-confidence does not come from the prior. The interval score is between 31.5 and 34.5 with no cycles seen, between 10.0 and 10.3 after 100 and between 42.3 and 43.4 after 300, which leaves the gain after 100 cycles and the loss after 300 where they were. The value of the retirement rule is the same to three decimals at $\alpha = 0.1$, 0.333 after 100 cycles and 0.462 after 300, and it moves by at most 0.04 at $\alpha = 0.2$ and at $\alpha = 0.5$, which is within the differences that the text called not meaningful. The data fix the precision at about 1,800 whatever the prior, so these results follow from the model of the residuals and not from the prior of one parameter.

In [26]:
#| output: false
#| code-summary: "Show the numbers quoted in the text"
# Numbers that the text quotes. This cell prints them and is hidden on the page.
posterior = fits["partial"].posterior
print(f"Prior predictive SoH percentiles 5/50/95: {np.percentile(prior_soh, [5, 50, 95]).round(1)}; observed {np.percentile(train['soh'], [5, 50, 95]).round(1)}")
for quantity, draws in (("log rate", posterior["loglambda_chem"]), ("amplitude", posterior["amplitude_chem"]), ("alpha", posterior["alpha_chem"])):
    first, second = (draws.sel(chemistry=name).to_numpy().ravel() for name in CHEMISTRIES)
    print(f"{quantity}: {CHEMISTRIES[0]} {first.mean():.2f} ({np.percentile(first, 5):.2f} to {np.percentile(first, 95):.2f}), {CHEMISTRIES[1]} {second.mean():.2f} ({np.percentile(second, 5):.2f} to {np.percentile(second, 95):.2f}), P({CHEMISTRIES[1]} > {CHEMISTRIES[0]}) = {(second > first).mean():.3f}")
print("beta (mean, sd):", {name: (round(float(posterior['beta'].sel(feature=name).mean()), 3), round(float(posterior['beta'].sel(feature=name).std()), 3)) for name in FEATURES})
print("phi mean", round(float(posterior["phi"].mean()), 1), "| sigma_cell_alpha", posterior["sigma_cell_alpha"].mean(("chain", "draw")).to_numpy().round(2), "| sigma_cell_lambda", posterior["sigma_cell_lambda"].mean(("chain", "draw")).to_numpy().round(2))
print("prior P(phi > posterior mean):", round(float((np.exp(prior_many.prior["phi"].to_numpy().ravel()) if False else prior_many.prior["phi"].to_numpy().ravel() > float(posterior["phi"].mean())).mean()), 4))
control = az.from_netcdf(CACHE / "train-control.nc")
print("control (complete pooling, no diagnostics): phi mean", round(float(control.posterior["phi"].mean()), 1))
print("mean absolute distance between no pooling and partial pooling:", estimates.pivot_table(index=["parameter", "cell"], columns="method", values="mean").assign(gap=lambda rows: (rows["No pooling"] - rows["Partial pooling"]).abs()).groupby("parameter")["gap"].mean().round(2).to_dict())
print(scores.round(3).to_string(index=False))
print(diagnostics.round(3).to_string(index=False))
for ratio in (0.1, 0.2, 0.3, 0.5):
    row = values[np.isclose(values["ratio"], ratio)].pivot_table(index="strategy", columns="cycles_seen", values="expense").round(3)
    print(f"expense at ratio {ratio} (units of the loss):"); print(row.to_string())
    row = values[np.isclose(values["ratio"], ratio)].pivot_table(index="strategy", columns="cycles_seen", values="value").round(3)
    print(f"value at ratio {ratio}:"); print(row.to_string())
print("sensitivity:"); print(sensitivity.round(3).to_string(index=False))
for setting, idata in sensitivity_fits.items():
    p = idata.posterior["loglambda_chem"].stack(s=("chain", "draw"))
    print(setting, "P(CX2 rate > CS2 rate) =", round(float((p.sel(chemistry=CHEMISTRIES[1]) > p.sel(chemistry=CHEMISTRIES[0])).mean()), 3), "| phi mean", round(float(idata.posterior["phi"].mean()), 1))

Prior predictive SoH percentiles 5/50/95: [ 23.6  91.1 109.1]; observed [50.8 81.9 97.7]
log rate: CS2 -0.31 (-0.78 to 0.30), CX2 0.73 (0.44 to 1.05), P(CX2 > CS2) = 0.992
amplitude: CS2 2.76 (2.33 to 3.29), CX2 1.03 (1.01 to 1.06), P(CX2 > CS2) = 0.000
alpha: CS2 1.19 (1.08 to 1.32), CX2 1.98 (1.72 to 2.22), P(CX2 > CS2) = 1.000
beta (mean, sd): {'charge_current_auc': (-0.217, 0.142), 'charge_current_mean': (-0.182, 0.142), 'discharge_voltage_auc': (0.362, 0.016), 'discharge_voltage_crest': (-0.009, 0.012)}
phi mean 1810.3 | sigma_cell_alpha [0.14 0.34] | sigma_cell_lambda [0.52 0.47]
prior P(phi > posterior mean): 0.002
control (complete pooling, no diagnostics): phi mean 34.7
mean absolute distance between no pooling and partial pooling: {'Initial level (logit)': 0.24, 'Log rate (per 1,000 cycles)': 0.4}
strategy  cycles_seen   mae   rmse  coverage  width  interval   n
    none            0 6.645 10.632     1.000 73.635    73.635 222
    none          100 1.561  2.374     1.000 22.0

## Limits of the Study

The test is small: two cells, one of each chemistry, are held out, and every coverage figure rests on two curves, so a different pair, or a cross-validation over cells, would give a firmer answer. The retirement decision rests on the same two cells and on 222 inspections of them, which are neighbouring cycles of two lives and not independent, so the curves of @fig-value are uneven and differences of a few hundredths in value are not meaningful. The scoring treats every cycle as an inspection at which a cell can be retired, whereas a real operator retires a cell once, and 64.9% of the scored inspections are below the threshold, a share that belongs to the window after cycle 350 and not to a fleet in service. The costs are stated only through their ratio, and the value of a model depends on that ratio, as the last section shows. The residuals are treated as independent, although the over-confidence after 300 cycles points to correlation within a cell, and a model with an autoregressive error or a cell-level smooth term would be the next step.

The prior of the precision $\phi$ is tight and the posterior lies far in its upper tail, and the refits of the last section show that neither the chemistries nor the scores of partial pooling depend on it. Only partial pooling was refitted, so the comparison with complete pooling rests on the default prior for both. Two chemistries are few, because the spread between chemistries is estimated from two groups, so the chemistry-level standard deviations are set partly by their priors and are not interpreted here. The curve is simple, since a saturating exponential with a linear effect of the diagnostics has no knee, which the end of life of these cells shows, and every eighth cycle is used, which shortens the run and discards little, although the observations that remain are still correlated.

## Summary of Findings

A hierarchical model with cells in chemistries separates what belongs to a chemistry from what belongs to a cell. The two chemistries differ in all three curve parameters, with a probability of 0.992 that CX2 has the higher rate, and the conclusion does not change under a narrower or a wider prior for the precision. Partial pooling also steadies the cell whose early life is missing, CX2_16, whose initial level falls from 3.54 to 1.96.

As a predictor for new cells, the hierarchy is better than one common curve in a single setting. After 100 cycles it gave the same coverage as complete pooling, 94.6% against 95.0%, with an interval score of 10.3 against 18.7. With no cycles seen it was too wide, with an interval score of 31.5 against 18.6, and after 300 cycles it was over-confident, with a coverage of 52% and an interval score of 43.3. The reason is that one diagnostic nearly determines the state of health here, and the observation model treats successive residuals as independent.

For an operator who must decide whether to retire a cell, the same results become a rule with a cost. With no cycle of a new cell seen, the hierarchy has no value at ratios up to 0.2, because its intervals are too wide to separate any cell from the rest. After 300 cycles it has the highest value of the three strategies for an operator for whom a failure costs at least three times a retirement, with 0.46 against 0.12 for one common curve at a ratio of 0.1, and after 100 cycles only for an operator for whom a failure costs ten times as much. For an operator with balanced costs the sharper probabilities add nothing, and a vague model does better. Where the question concerns cells and chemistries, the hierarchy answers it. Where it is prediction or retirement, it should be compared with complete pooling and with a control that has no diagnostics before it is adopted, in the way done here, and the comparison should be made at the cost ratio of the operator.

## Reproducible Results

The notebook is in the [bayesian-modelling](https://github.com/sambaiga/bayesian-modelling) repository and runs from the first cell to the last in the `uv` environment of the repository, with PyMC 5.25, ArviZ 0.22 and lets-plot 4.11, and downloads the feature file when it first runs. The model, the sampling, the scores and the figures are functions of the `bayes` package, which has unit tests, and the cells of the notebook only call them. The sampling settings are in the first cell, and setting the environment variable `HIER_FAST=1` runs the whole notebook with a few draws in a minute or two, to check that it works before the full run. The fits are stored in `cache/`, so a repeat of the full run reads them back.